# 01 — Raw EDA & Subject-wise Split (multi-person test)

**Mục tiêu:** đọc ZIP YOLO11n đã phục hồi, kiểm tra 13 upper-body joints, tạo split theo **danh sách người test**, và xem **ảnh thật từ video** của từng nhãn. CPU-only, không normalize, không train.

**Đầu ra:** `outputs/raw_eda/experiments/<test_people>/` và một ZIP review gọn (CSV, JSON, PNG; không có video).

## Cell 1 — Drive, dataset và danh sách TEST_PEOPLE
Cấu hình danh sách người test tại đây; ví dụ `["TRUONG"]` hoặc `["TRUONG", "KIET"]`. Những người còn lại thuộc `train_pool`; **chưa chia validation**. Mỗi danh sách có thư mục output riêng để không ghi đè split cũ.

In [ ]:
from pathlib import Path
import os, sys

# Mount Drive trước khi xác định ROOT (tránh lỗi lần chạy đầu ở Colab).
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass

# Colab: dùng đúng root Drive; local: tự tìm project root từ thư mục chạy notebook.
COLAB_ROOT = Path('/content/drive/MyDrive/KI_8/IMP302/LAB_2/truongedu')
ROOT = Path(os.environ['POSEFORGE_ROOT']).expanduser() if os.environ.get('POSEFORGE_ROOT') else (
    COLAB_ROOT if COLAB_ROOT.is_dir() else next(
        (p for p in (Path.cwd(), *Path.cwd().parents)
         if (p / 'notebooks').is_dir() and (p / 'shared').is_dir()), Path.cwd()
    )
)
ROOT = ROOT.resolve()
# GIỮ NGUYÊN PATH DATASET HIỆN TẠI
RAW_ZIP = Path(os.environ.get('POSEFORGE_ZIP', str(ROOT / 'raw_dataset_zip/Real_Posture_Dataset_YOLO11n_singleperson_v2.zip')))
BASE_OUT = Path(os.environ.get('POSEFORGE_OUT', str(ROOT / 'outputs/raw_eda')))
TEST_PEOPLE = ['TRUONG']  # Ví dụ nhiều người: ['TRUONG', 'KIET']; [] = AUTO 1 người
SPLIT_SEED = 302
CONF_THRESHOLD = 0.35  # Phân tích visibility, KHÔNG lọc training ở Notebook 01

assert isinstance(TEST_PEOPLE, list), 'TEST_PEOPLE phải là list, ví dụ ["TRUONG", "KIET"]'
assert len(TEST_PEOPLE) == len(set(TEST_PEOPLE)), 'TEST_PEOPLE có ID trùng'
RUN_TAG = 'test_' + ('AUTO_seed' + str(SPLIT_SEED) if not TEST_PEOPLE else '_'.join(sorted(TEST_PEOPLE)))
OUT = BASE_OUT / 'experiments' / RUN_TAG
assert RAW_ZIP.is_file(), f'Không tìm thấy ZIP: {RAW_ZIP}'
OUT.mkdir(parents=True, exist_ok=True)
# Đặt ROOT (local/Colab/Kaggle) vào Python path để import package shared.
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from shared.dataset_pipeline import scan_raw_zip, subject_split, file_sha256, COCO13
print('Input:', RAW_ZIP)
print('Output:', OUT)
print('Test people:', TEST_PEOPLE or '[AUTO]')
print('CPU only')

## Cell 2 — Copy ZIP vào runtime (thanh tiến độ)
Copy file ZIP từ Drive sang ổ tạm `/content` để đọc nhanh hơn. **Không giải nén video, không sửa file gốc.** Đọc ZIP từ runtime cho nhanh; dữ liệu nguồn vẫn ở Drive.

In [ ]:
import hashlib, tempfile
from tqdm.auto import tqdm

LOCAL_ZIP = Path('/content/poseforge_raw_176clips.zip') if Path('/content').exists() else Path(tempfile.gettempdir()) / 'poseforge_raw_local.zip'
source_size = RAW_ZIP.stat().st_size
hash_sha = hashlib.sha256()
if LOCAL_ZIP.resolve() == RAW_ZIP.resolve():
    source_sha = file_sha256(RAW_ZIP)
else:
    with RAW_ZIP.open('rb') as src, LOCAL_ZIP.open('wb') as dst, tqdm(total=source_size, unit='B', unit_scale=True, desc='Copy dataset ZIP') as bar:
        while chunk := src.read(4 * 1024 * 1024):
            dst.write(chunk)
            hash_sha.update(chunk)
            bar.update(len(chunk))
    source_sha = hash_sha.hexdigest()
assert LOCAL_ZIP.stat().st_size == source_size
print('SHA-256:', source_sha)
print('Staged ZIP:', LOCAL_ZIP)

## Cell 3 — Parse & validate dữ liệu gốc (thanh tiến độ)
Đọc `labels.csv`, `metadata.json`, timestamps và từng dòng JSONL của raw YOLO17. Một frame có **mảng 17×3** chưa chắc có đủ **13 keypoints hợp lệ**. Hai khái niệm được đếm riêng.

In [ ]:
import json
import numpy as np
import pandas as pd
from IPython.display import display

catalog, frame_quality, joint_coverage, examples, audit = scan_raw_zip(LOCAL_ZIP, CONF_THRESHOLD)
people = sorted(catalog.person_id.unique())
takes = catalog.take_id.nunique()
print(f'Clip-camera: {len(catalog)} | Takes: {takes} | Persons: {len(people)} | Labels: {catalog.label.nunique()}')
print(f'YOLO17 shape OK: {int(catalog.pose17_frames.sum()):,}/{int(catalog.frames.sum()):,}')
print(f'Full 13/13: {int(catalog.full13_frames.sum()):,} | Partial: {int(catalog.partial13_frames.sum()):,} | No usable: {int(catalog.none13_frames.sum()):,}')
print('Checkpoint provenance:', audit['model_counts'])
print('Frame statuses:', audit['status_counts'])
print('Structural issues:', len(audit['issues']))
for issue in audit['issues'][:12]: print('[ISSUE]', issue)
display(catalog[['person_id','camera_id','label','frames','pose17_rate','full13_rate','timebase_discrepancy']].head(8))
assert len(catalog) == 176 and takes == 88 and len(people) == 13 and catalog.label.nunique() == 6, 'Dataset count differs from Phase 1 preflight; stop here.'
assert not audit['issues'], 'Unexpected data format/provenance; inspect issues before split.'
assert catalog.pose_model.eq('yolo11n-pose.pt').all(), 'Mixed extractor checkpoints detected'
assert int(catalog.frames.sum()) == len(frame_quality), 'Missing or duplicate frame rows'

## Cell 4 — Tóm tắt chất lượng raw
Thống kê theo **take** (không nhân đôi hai camera), clip-camera và frame; quan sát tỉ lệ đầy đủ 13 điểm, trạng thái detection và timebase. Chưa loại frame nào.

In [ ]:
take_table = catalog.drop_duplicates('take_id')
raw_summary = {
    'source_zip_sha256': source_sha,
    'clip_camera_count': int(len(catalog)),
    'take_count': int(takes),
    'people_count': int(len(people)),
    'labels': audit['label_names'],
    'total_frames': int(catalog.frames.sum()),
    'pose17_shape_ok_frames': int(catalog.pose17_frames.sum()),
    'full13_frames': int(catalog.full13_frames.sum()),
    'partial13_frames': int(catalog.partial13_frames.sum()),
    'none13_frames': int(catalog.none13_frames.sum()),
    'timebase_discrepancies': int(catalog.timebase_discrepancy.sum()),
    'model_counts': audit['model_counts'],
    'status_counts': audit['status_counts'],
    'quality_conf_threshold': CONF_THRESHOLD,
    'issues': audit['issues'],
}
print('Takes by label:')
display(take_table.groupby('label').agg(takes=('take_id','size'), people=('person_id','nunique')))
print('Pose coverage by camera:')
display(catalog.groupby('camera_id').agg(clips=('clip_id','size'), frames=('frames','sum'), pose17=('pose17_frames','sum'), full13=('full13_frames','sum')))
print('Camera status vs take status:')
display(pd.crosstab(catalog.status_camera, catalog.status_take))

## Cell 5 — EDA: số lượng, độ phủ và phân bố tư thế
Biểu đồ được **hiện trực tiếp trong notebook và lưu PNG** vào `outputs/raw_eda/experiments/<test_people>/charts/`. Đơn vị take, clip-camera, frame được ghi tách biệt để tránh hiểu nhầm.

In [ ]:
import matplotlib.pyplot as plt

CHARTS = OUT / 'charts'
CHARTS.mkdir(exist_ok=True)
plt.rcParams.update({'figure.dpi': 110, 'savefig.dpi': 130, 'font.size': 10})

def show_save(fig, filename):
    fig.tight_layout()
    fig.savefig(CHARTS / filename, bbox_inches='tight')
    plt.show()
    plt.close(fig)

# 1. Take count by class (unique take, not clips)
take_counts = take_table.label.value_counts().reindex(range(6), fill_value=0)
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.bar([f'L{i}' for i in take_counts.index], take_counts.values)
ax.set(title='Unique takes by label', xlabel='Posture class', ylabel='Takes')
for i, v in enumerate(take_counts): ax.text(i, v, str(v), ha='center', va='bottom')
show_save(fig, '01_takes_by_label.png')

# 2. Person x label (unique takes)
coverage = pd.crosstab(take_table.person_id, take_table.label).reindex(columns=range(6), fill_value=0)
fig, ax = plt.subplots(figsize=(9, 5))
im = ax.imshow(coverage.to_numpy(), aspect='auto', cmap='Blues')
ax.set(xticks=range(6), xticklabels=[f'L{i}' for i in range(6)], yticks=range(len(coverage)), yticklabels=coverage.index, title='Subject × label: unique takes')
for (i, j), v in np.ndenumerate(coverage.to_numpy()): ax.text(j, i, str(v), ha='center', va='center', color='white' if v >= 3 else 'black', fontsize=8)
fig.colorbar(im, ax=ax, label='Takes')
show_save(fig, '02_subject_label_heatmap.png')

# 3. Clip / frame distributions by label
frame_counts = catalog.groupby('label').frames.sum().reindex(range(6), fill_value=0)
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.bar([f'L{i}' for i in frame_counts.index], frame_counts.values)
ax.set(title='Raw frames by label (both cameras)', xlabel='Posture class', ylabel='Frames')
show_save(fig, '03_frames_by_label.png')

# 4. Raw detection statuses
statuses = pd.Series(audit['status_counts']).sort_values(ascending=True)
fig, ax = plt.subplots(figsize=(8, max(3.2, len(statuses)*.45)))
ax.barh(statuses.index, statuses.values)
ax.set(title='Raw YOLO frame statuses', xlabel='Frames')
show_save(fig, '04_detection_status.png')

# 5. Number of usable upper-body joints per frame
quality = frame_quality.valid13_count.value_counts().reindex(range(14), fill_value=0)
fig, ax = plt.subplots(figsize=(9, 3.5))
ax.bar(quality.index, quality.values)
ax.set(title=f'Usable upper-body joints per frame (confidence ≥ {CONF_THRESHOLD})', xlabel='Valid joints (0–13)', ylabel='Frames', xticks=range(14))
show_save(fig, '05_valid13_histogram.png')

## Cell 6 — EDA: visibility 13 joints, camera và thời gian
Kiểm tra điểm nào thường bị che, khác biệt giữa 2 camera và chênh lệch thời gian MP4–capture. **Chênh lệch thời gian không tự động được coi là frame lỗi.**

In [ ]:
# 6. Joint visibility by class (13 keypoints)
view = joint_coverage[joint_coverage.label != 'all']
coverage_joint = view.groupby(['label','joint'])[['valid_frames','frames']].sum()
coverage_joint['rate'] = coverage_joint.valid_frames / coverage_joint.frames
mat = coverage_joint.rate.unstack('joint').reindex(index=range(6), columns=COCO13)
fig, ax = plt.subplots(figsize=(12, 4.2))
im = ax.imshow(mat, vmin=0, vmax=1, cmap='YlGnBu', aspect='auto')
ax.set(xticks=range(13), xticklabels=COCO13, yticks=range(6), yticklabels=[f'L{i}' for i in range(6)], title='13-joint visibility by label')
plt.setp(ax.get_xticklabels(), rotation=40, ha='right')
fig.colorbar(im, ax=ax, label='Valid fraction')
show_save(fig, '06_joint_visibility_by_label.png')

# 7. Frames with raw17 pose / full13 by camera
cam = catalog.groupby('camera_id')[['frames','pose17_frames','full13_frames']].sum()
cam['pose17_pct'] = 100 * cam.pose17_frames / cam.frames
cam['full13_pct'] = 100 * cam.full13_frames / cam.frames
fig, ax = plt.subplots(figsize=(7.5, 3.5))
x = np.arange(len(cam)); ax.bar(x - .18, cam.pose17_pct, width=.36, label='17×3 array present')
ax.bar(x + .18, cam.full13_pct, width=.36, label='All 13 valid')
ax.set(xticks=x, xticklabels=cam.index, ylim=(0, 105), ylabel='% frames', title='Pose coverage by camera')
ax.legend()
show_save(fig, '07_camera_pose_coverage.png')

# 8. Take duration distribution from metadata
fig, ax = plt.subplots(figsize=(7.5, 3.5))
ax.hist(take_table.duration_s, bins=15)
ax.set(title='Take duration distribution (manifest/capture)', xlabel='Seconds', ylabel='Takes')
show_save(fig, '08_take_durations.png')

# 9. Encoded MP4 duration / captured timeline ratio
ratios = catalog.encoded_to_capture_ratio.replace([np.inf,-np.inf], np.nan).dropna()
fig, ax = plt.subplots(figsize=(7.5, 3.5))
ax.hist(ratios, bins=20)
ax.axvline(.85, color='red', linestyle='--', linewidth=1)
ax.axvline(1.15, color='red', linestyle='--', linewidth=1)
ax.set(title='MP4 timebase / capture timebase ratio', xlabel='Duration ratio', ylabel='Clip-camera records')
show_save(fig, '09_timebase_ratio.png')
print(f'Timebase discrepancy (>15%): {int(catalog.timebase_discrepancy.sum())}/{len(catalog)}')

## Cell 7 — Raw skeleton: 6 nhãn (không đọc hình từ video)
Hiển thị một ví dụ keypoints cho mỗi nhãn, giữ **tọa độ pixel gốc**. Đây là kiểm tra hình học nhanh, không đại diện cho toàn bộ video hoặc chất lượng nhãn.

In [ ]:
# COCO body links among first 13 points
EDGES = [(0,1),(0,2),(1,3),(2,4),(5,6),(5,7),(7,9),(6,8),(8,10),(5,11),(6,12),(11,12)]
fig, axes = plt.subplots(2, 3, figsize=(12, 7))
for label, ax in enumerate(axes.flat):
    item = examples.get(label)
    if item is None:
        ax.text(.5,.5,'No suitable pose',ha='center',va='center',transform=ax.transAxes)
        ax.axis('off'); continue
    kp = np.asarray(item['keypoints']); valid = (kp[:,2] >= CONF_THRESHOLD)
    valid &= (kp[:,0]>=0)&(kp[:,0]<item['width'])&(kp[:,1]>=0)&(kp[:,1]<item['height'])
    for a,b in EDGES:
        if valid[a] and valid[b]: ax.plot(kp[[a,b],0], kp[[a,b],1], color='steelblue', linewidth=2)
    ax.scatter(kp[valid,0], kp[valid,1], s=18, color='darkorange', zorder=3)
    ax.set(xlim=(0,item['width']), ylim=(item['height'],0), aspect='equal', title=f'L{label} • {item["person_id"]} / {item["camera_id"]}')
    ax.set_xlabel('x (pixels)'); ax.set_ylabel('y (pixels)')
show_save(fig, '10_raw_pose_examples.png')
print('Note: video frames are NOT copied into this report.')

## Cell 8 — Khóa subject-wise split (nhiều người test)
**Test:** những người có ID trong `TEST_PEOPLE`; **Train pool:** tất cả người còn lại. Validation chỉ được lấy từ train pool ở bước training. Nếu split đã tồn tại, notebook kiểm tra checksum nguồn và danh sách test, không âm thầm đổi split.

In [ ]:
SPLIT_FILE = OUT / 'split_manifest.json'
requested = TEST_PEOPLE or ['AUTO']
if SPLIT_FILE.exists():
    previous = json.loads(SPLIT_FILE.read_text(encoding='utf-8'))
    assert previous['source_zip_sha256'] == source_sha, 'ZIP đã đổi: lưu kết quả vào experiment khác, không dùng split cũ.'
    assert previous['seed'] == SPLIT_SEED, 'Seed đã thay đổi; phải dùng experiment khác.'
    if TEST_PEOPLE:
        assert sorted(TEST_PEOPLE) == previous['test_people'], 'TEST_PEOPLE khác split đã khóa.'
    print('Reusing LOCKED test people:', previous['test_people'])
    requested = previous['test_people']

split_df, split_info = subject_split(catalog, test_people=requested, seed=SPLIT_SEED)
split_info['source_zip_sha256'] = source_sha
assert len(split_info['train_pool_people']) + len(split_info['test_people']) == len(people)
assert split_df.clip_id.nunique() == 176
assert set(split_df.loc[split_df.split=='test', 'person_id']).isdisjoint(
    set(split_df.loc[split_df.split=='train_pool', 'person_id']))

print('TEST (không dùng để tune model):', ', '.join(split_info['test_people']))
print('TRAIN POOL:', len(split_info['train_pool_people']), 'người:', ', '.join(split_info['train_pool_people']))
print('Validation: CHƯA FIX — chỉ chọn từ train pool ở bước training')
display(pd.crosstab(split_df.split, split_df.label))
plot_split = pd.crosstab(split_df.split, split_df.label).reindex(index=['train_pool','test'], columns=range(6), fill_value=0)
fig, ax = plt.subplots(figsize=(7, 3.2))
plot_split.T.plot(kind='bar', ax=ax, width=.8)
ax.set(title='Clip-camera counts by split & label', xlabel='Label', ylabel='Clip-camera records', xticklabels=[f'L{i}' for i in range(6)])
show_save(fig, '11_split_label_coverage.png')

## Cell 9 — Ảnh thật từ video: 1 frame × 6 nhãn cho mỗi người TEST
Lấy frame gốc từ MP4 **bên trong ZIP** (không sử dụng ảnh giả), vẽ 13 keypoints/bbox đã lưu. Mỗi người test có một bảng 2×3. Các MP4 được giải nén **tạm thời trong runtime**, không lưu lên Drive; chỉ lưu PNG và CSV lựa chọn để kiểm tra lại.

In [ ]:
import cv2
import shutil
import tempfile
from zipfile import ZipFile
from tqdm.auto import tqdm

# Dùng frame gần giữa clip, ưu tiên điểm có nhiều keypoints hợp lệ.
frame_candidates = frame_quality.merge(
    catalog[['clip_id', 'video_path', 'raw_member', 'frames']], on='clip_id', validate='many_to_one'
)
frame_candidates['middle_distance'] = (
    frame_candidates.frame_index - (frame_candidates.frames - 1) / 2
).abs()
EDGES = [(0,1),(0,2),(1,3),(2,4),(5,6),(5,7),(7,9),(6,8),(8,10),(5,11),(6,12),(11,12)]

review_frames = []
with ZipFile(LOCAL_ZIP) as zf, tempfile.TemporaryDirectory() as tmp:
    prefix = next(n[:-len('labels.csv')] for n in zf.namelist()
                  if n.endswith('Real_Posture_Dataset/labels.csv'))
    tasks = [(p, label) for p in split_info['test_people'] for label in range(6)]
    figures = {}
    for person in split_info['test_people']:
        figures[person] = plt.subplots(2, 3, figsize=(14, 8))
    for person, label in tqdm(tasks, desc='Load real test frames', unit='sample'):
        fig, axes = figures[person]
        ax = axes.flat[label]
        sub = frame_candidates[(frame_candidates.person_id == person) &
                               (frame_candidates.label == label) &
                               (frame_candidates.pose17_shape_ok == 1)].copy()
        if sub.empty:
            ax.text(.5, .5, 'No pose frame', ha='center', va='center', transform=ax.transAxes)
            ax.axis('off')
            continue
        best = sub.sort_values(['valid13_count', 'middle_distance'], ascending=[False, True]).iloc[0]
        frame_index = int(best.frame_index)
        member = prefix + best.video_path
        video_temp = Path(tmp) / ('selected_' + str(len(review_frames)) + '.mp4')
        with zf.open(member) as src, video_temp.open('wb') as dst:
            shutil.copyfileobj(src, dst)
        cap = cv2.VideoCapture(str(video_temp))
        cap.set(cv2.CAP_PROP_POS_FRAMES, frame_index)
        ok, bgr = cap.read()
        cap.release()
        video_temp.unlink(missing_ok=True)
        assert ok and bgr is not None, f'MP4 failed to decode {member}, frame {frame_index}'

        item = None
        with zf.open(prefix + best.raw_member) as src:
            for line in src:
                if not line.strip():
                    continue
                record = json.loads(line)
                if int(record['frame_index']) == frame_index:
                    item = record
                    break
        assert item is not None, f'Missing JSONL frame {frame_index} for {best.clip_id}'
        keypoints = np.asarray(item['keypoints'], dtype=np.float32)[:13]
        h, w = bgr.shape[:2]
        visible = (keypoints[:, 2] >= CONF_THRESHOLD) & np.isfinite(keypoints).all(axis=1)
        visible &= (keypoints[:,0]>=0)&(keypoints[:,0]<w)&(keypoints[:,1]>=0)&(keypoints[:,1]<h)
        # Draw the stored bbox (not a manually created detector result).
        bbox = item.get('bbox')
        if bbox is not None and len(bbox) >= 4:
            x1,y1,x2,y2 = [int(v) for v in bbox[:4]]
            cv2.rectangle(bgr, (x1,y1), (x2,y2), (50,215,60), 2)
        for a, b in EDGES:
            if visible[a] and visible[b]:
                pa, pb = tuple(keypoints[a,:2].astype(int)), tuple(keypoints[b,:2].astype(int))
                cv2.line(bgr, pa, pb, (240,170,30), 2)
        for idx in np.flatnonzero(visible):
            cv2.circle(bgr, tuple(keypoints[idx,:2].astype(int)), 3, (30,130,240), -1)
        ax.imshow(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB))
        ax.set_title(f'L{label} | {best.camera_id} | frame {frame_index} | {int(visible.sum())}/13', fontsize=10)
        ax.axis('off')
        review_frames.append(dict(person_id=person, label=label, clip_id=best.clip_id,
                                  camera_id=best.camera_id, frame_index=frame_index,
                                  valid13_count=int(visible.sum()), video_path=best.video_path))
    for person, (fig, axes) in figures.items():
        fig.suptitle(f'REAL VIDEO samples — TEST person {person}', fontsize=14)
        show_save(fig, f'12_real_test_frames_{person}.png')

review_frames_df = pd.DataFrame(review_frames)
review_frames_df.to_csv(OUT / 'test_visual_samples.csv', index=False)
display(review_frames_df[['person_id', 'label', 'camera_id', 'frame_index', 'valid13_count']])
print(f'Visualized {len(review_frames)} actual video frames; no MP4 added to reports.')

## Cell 10 — Export reports và review ZIP nhỏ
Lưu CSV/JSON/PNG và một ZIP gọn để phân tích. Không chứa video, weights hay toàn bộ raw keypoints.

In [ ]:
import zipfile

catalog.to_csv(OUT / 'raw_catalog.csv', index=False)
frame_quality.to_csv(OUT / 'raw_frame_quality.csv.gz', index=False, compression='gzip')
joint_coverage.to_csv(OUT / 'joint_coverage.csv', index=False)
split_df.to_csv(OUT / 'split_manifest.csv', index=False)
(OUT / 'raw_summary.json').write_text(json.dumps(raw_summary, ensure_ascii=False, indent=2), encoding='utf-8')
(OUT / 'split_manifest.json').write_text(json.dumps(split_info, ensure_ascii=False, indent=2), encoding='utf-8')
(OUT / 'raw_issues.json').write_text(json.dumps(audit['issues'], ensure_ascii=False, indent=2), encoding='utf-8')

# Review file outside the experiment directory avoids including itself in ZIP.
REVIEW_ZIP = BASE_OUT / f'raw_eda_review_{RUN_TAG}.zip'
with zipfile.ZipFile(REVIEW_ZIP, 'w', compression=zipfile.ZIP_DEFLATED, compresslevel=6) as zf:
    for p in sorted(OUT.rglob('*')):
        if p.is_file():
            zf.write(p, arcname=str(Path('raw_eda') / p.relative_to(OUT)))
print('Saved:', OUT)
print('Review ZIP:', REVIEW_ZIP, f'({REVIEW_ZIP.stat().st_size / 1024**2:.2f} MB)')
print('Charts:', len(list(CHARTS.glob('*.png'))))
print('Split locked:', ', '.join(split_info['test_people']))
print('Notebook 01 PASS; ready for Notebook 02 preprocessing.')